# Week 6 Lab 03: Threshold Tuning for Cordwell Safety Alerts

**Scenario:** Cordwell Home and Hardware wants urgent safety tickets, a customer smelling gas, a railing that shifts, a wall plate hot to the touch, flagged for immediate triage while routine project and policy questions flow to the normal queue. You will train the classifier, and then spend most of the lab on the decision the model cannot make for you: where to put the cutoff.

**Estimated duration:** 120 minutes.

By the end of this lab you will be able to:

- Split data three ways and explain what the validation set is for when tuning begins.
- Evaluate a classifier at any decision threshold, not just the default 0.5.
- Sweep thresholds across the probability range and read the precision, recall, and F1 curves the sweep produces.
- Select operating thresholds for competing business goals and defend each choice with rows from your own sweep.
- Lock one threshold, grade it exactly once on the test set, and interpret the validation-to-test gap.

**How this closes Module 01.** Lab 01 taught what the four metrics mean and previewed thresholds on three values. Lab 02 showed which metric deserves to drive on imbalanced data and treated the cutoff as one of two fixes. This lab makes the cutoff the whole subject, done the way production teams do it: sweep on validation, choose per stakeholder, lock, and spend the test set exactly once. The validation set, which Lab 02 deliberately dropped, returns for a reason stated in Task 1: today we tune, and tuning needs somewhere to land that is not the test set.

## How this notebook works

Cells marked **PROVIDED** are plumbing: run them and move on. Cells marked **TASK** contain a function contract and a `raise NotImplementedError`; replace the raise with your implementation. Each task is followed by an **apply** cell that uses your function and a **checks** cell that grades it.

The check harness never crashes the notebook. Unwritten tasks print `[TODO]`, wrong answers print `[FAIL]` with the reason, and your running score appears wherever `summary()` is called. A fresh Run All on this notebook completes with zero errors and 6 of 27 checks passing; your goal is 27 of 27.

Two hint files sit next to this notebook. `HINTS.md` offers three escalating nudges per task. `HINTS_DETAILED.md` shows the working core of each task with commentary. Pick one tier per task; reading both wastes time.

## Part 0: Environment

Everything runs locally on CPU. No Docker services, no local LLM server, no network calls, and nothing here touches a GPU. Same pinned environment as Labs 1.1 and 1.2; machines set up this morning need nothing new.

If the imports cell fails, run `pip install -r requirements.txt` in your environment and restart the kernel.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# One seed drives corpus generation, the splits, and the model, so your
# numbers match the worked target output exactly. Each lab today declares
# its own seed, chosen empirically so the numbers teach well; this lab's
# seed makes the three stakeholder thresholds land on three different
# values, which is the situation worth practicing.
RANDOM_SEED = 11

print(f"scikit-learn {sklearn.__version__}")
print(f"pandas       {pd.__version__}")
print(f"numpy        {np.__version__}")

In [ ]:
CHECK_RESULTS = {}

def check(name: str, fn) -> None:
    """Run a zero-argument callable and record a named PASS or FAIL.

    Unimplemented tasks (NotImplementedError) and missing upstream results
    (NameError, or None placeholders being poked) report as TODO, not FAIL.
    """
    try:
        ok = bool(fn())
    except NotImplementedError:
        CHECK_RESULTS[name] = False
        print(f"[TODO] {name}: task not implemented yet")
        return
    except Exception as exc:
        CHECK_RESULTS[name] = False
        if isinstance(exc, NameError) or "NoneType" in str(exc):
            print(f"[TODO] {name}: depends on an earlier task")
        else:
            print(f"[FAIL] {name}: {type(exc).__name__}: {exc}")
        return
    CHECK_RESULTS[name] = ok
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")

def attempt(label: str, fn):
    """Run a student function for an apply cell; return its result or None."""
    try:
        return fn()
    except NotImplementedError:
        print(f"[TODO] {label}: implement the task above, then re-run this cell.")
        return None
    except Exception as exc:
        print(f"[ERROR] {label}: {type(exc).__name__}: {exc}")
        return None

def summary() -> None:
    total = len(CHECK_RESULTS)
    passed = sum(CHECK_RESULTS.values())
    print(f"Checks passing: {passed}/{total}")

print("Check harness ready.")

In [ ]:
check(
    "env: scikit-learn 1.6 or newer",
    lambda: tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 6),
)
check(
    "env: pandas 2.0 or newer",
    lambda: tuple(int(p) for p in pd.__version__.split(".")[:2]) >= (2, 0),
)
check(
    "env: numpy 2.0 or newer",
    lambda: tuple(int(p) for p in np.__version__.split(".")[:2]) >= (2, 0),
)
summary()

## Part 1: The ticket corpus (provided)

Assembling a corpus was Lab 1.2's Task 1; today it is plumbing, because today's lesson is the cutoff, not the dataset. Still, read the generator before running past it, because one design choice below is what makes the threshold sweep worth doing.

Urgent tickets are not all equally loud. Some customers write three or four alarming sentences; others bury a single line about a faint gas smell inside a paragraph of paint questions. The generator models that spectrum directly: each urgent ticket draws between 1 and 4 safety sentences (weighted toward the quiet end) and pads the rest with routine chatter. Some routine tickets, about one in six, also quote one safety-flavored sentence without describing an active emergency of their own, the way a customer asks about a product because of something that happened to a neighbor.

The consequence, which you will see the moment you sweep: the model's probability scores spread across the whole range instead of piling up at 0 and 1. Loud tickets score high, quiet ones score in the murky middle, and where you put the cutoff genuinely decides who gets a safety callback. A corpus of only loud tickets would make every threshold from 0.1 to 0.9 behave identically, and this lab would have nothing to teach.

In [ ]:
SAFETY_SENTENCES = [
    "I smell gas near the water heater and I am not sure it is safe to stay in the house.",
    "The breaker keeps tripping every time I turn on the new table saw in the garage.",
    "There was a small spark and a burning smell when I plugged in the new extension cord.",
    "The ladder feels unstable on the deck and I am worried it could slip while I am painting.",
    "The ceiling fan box is loose and the fan wobbles badly whenever we turn it on.",
    "Water is dripping near an electrical outlet after we tiled the shower wall.",
    "A step on the new staircase kit is cracked and feels like it could give way.",
    "The pilot light on the gas heater keeps going out and I hear a faint hissing sound.",
    "After installing the new dimmer switch, the wall plate feels hot to the touch.",
    "The heavy cabinet we anchored to the wall is pulling away from the studs.",
    "The smoke detector chirped and went silent after the attic insulation was blown in.",
    "The deck railing shifts noticeably when anyone leans against it near the stairs.",
]

PROJECT_SENTENCES = [
    "We are planning a kitchen remodel and need advice on cabinet finishes and layouts.",
    "What is the difference between luxury vinyl plank and laminate flooring for a basement?",
    "Do you have sample boards for interior paint colors that we can take home?",
    "We want to build a raised garden bed and need to know which lumber holds up outdoors.",
    "Can you recommend an underlayment for installing laminate over a concrete slab?",
    "We are comparing quartz and butcher block countertops for a small kitchen island.",
    "Do you offer installation services for bathroom vanities and faucets?",
    "What size nails should we use for installing new baseboards in the living room?",
    "Do you carry pre-hung exterior doors with built-in blinds between the glass?",
    "We need help estimating how many deck boards to order for a 12 by 16 foot deck.",
]

POLICY_SENTENCES = [
    "What is the return policy for opened paint cans if the color is slightly off?",
    "Are there any weekend sales coming up on power tools or storage systems?",
    "Do you offer military discounts or price matching with other home centers?",
    "What are the store hours on Sundays for the lumber and garden departments?",
    "Can we schedule a design consultation for a small bathroom project online?",
    "Is curbside pickup available for large items like doors and sheet goods?",
    "How long do special orders for custom blinds usually take to arrive?",
    "Do you rent tools like tile saws and floor sanders by the day or by the hour?",
    "Are there any upcoming DIY workshops for decking or fencing projects?",
    "Can we use store credit cards and gift cards in the same transaction?",
]

def make_ticket(rng: random.Random, label: int) -> str:
    """Write one ticket. Urgent tickets sit on a loudness spectrum."""
    filler = PROJECT_SENTENCES + POLICY_SENTENCES
    if label == 1:
        # 1 to 4 safety sentences, weighted toward the quiet end: this
        # spread is what gives the probability scores their range.
        n_safety = rng.choice([1, 1, 2, 2, 3, 4])
        n_filler = rng.randint(3, 5)
        sentences = rng.sample(SAFETY_SENTENCES, n_safety) + rng.choices(filler, k=n_filler)
    else:
        n_filler = rng.randint(5, 7)
        sentences = rng.choices(filler, k=n_filler)
        # Hard negatives: about one routine ticket in six quotes a single
        # safety-flavored line without an active emergency of its own.
        if rng.random() < 0.18:
            sentences.append(rng.choice(SAFETY_SENTENCES))
    rng.shuffle(sentences)
    return " ".join(sentences)

def build_corpus(n_docs: int = 500, positive_fraction: float = 0.15, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Generate the Cordwell safety-alert corpus: 15 percent urgent."""
    rng = random.Random(seed)
    n_pos = round(n_docs * positive_fraction)
    rows = []
    for i in range(n_docs):
        label = 1 if i < n_pos else 0
        rows.append({"text": make_ticket(rng, label), "label": label})
    rng.shuffle(rows)
    return pd.DataFrame(rows)

print("Generator ready.")

In [ ]:
tickets_df = build_corpus()

print(f"Corpus size: {len(tickets_df)}")
print()
print("Class distribution (percent):")
dist = (tickets_df["label"].map({0: "routine", 1: "urgent_safety"})
        .value_counts(normalize=True) * 100).round(1)
print(dist.to_string())
print()
print("A loud urgent ticket and a quiet one, for contrast:")
urgent = tickets_df[tickets_df["label"] == 1]["text"]
by_mentions = urgent.str.count("I smell gas|burning smell|hissing|hot to the touch|give way|pulling away|wobbles|unstable|shifts|chirped|dripping|tripping")
print(" LOUD :", urgent[by_mentions.idxmax()][:220], "...")
print(" QUIET:", urgent[by_mentions.idxmin()][:220], "...")

In [ ]:
check(
    "corpus: 500 rows with columns text and label",
    lambda: tickets_df.shape == (500, 2) and list(tickets_df.columns) == ["text", "label"],
)
check(
    "corpus: exactly 75 urgent tickets (15 percent of 500)",
    lambda: int(tickets_df["label"].sum()) == 75,
)
check(
    "corpus: same seed reproduces the same corpus",
    lambda: build_corpus().equals(build_corpus()),
)
summary()

## Worked target output

Everything is seeded, so a correct implementation reproduces these numbers exactly. Code toward this target; if a check fails, compare your output here first.

**Split (Task 1):** 300 train, 100 validation, 100 test, stratified: 45, 15, and 15 urgent tickets respectively (15.0 percent in each split).

**Default threshold, validation (Tasks 2 and 3):** at the default 0.5, the pipeline flags only 7 of the 100 validation tickets.

```
Confusion matrix (rows true, columns predicted):
[[85  0]
 [ 8  7]]
accuracy   0.920
precision  1.000
recall     0.467
f1         0.636
```

**Threshold sweep on validation (Task 4):**

```
 threshold  accuracy  precision  recall    f1
      0.05      0.31      0.179   1.000 0.303
      0.10      0.81      0.441   1.000 0.612
      0.15      0.86      0.517   1.000 0.682
      0.20      0.94      0.737   0.933 0.824
      0.25      0.97      0.875   0.933 0.903
      0.30      0.96      1.000   0.733 0.846
      0.35      0.94      1.000   0.600 0.750
      0.40      0.94      1.000   0.600 0.750
      0.45      0.92      1.000   0.467 0.636
      0.50      0.92      1.000   0.467 0.636
      0.55      0.90      1.000   0.333 0.500
      0.60      0.88      1.000   0.200 0.333
      0.65      0.88      1.000   0.200 0.333
      0.70      0.87      1.000   0.133 0.235
      0.75      0.87      1.000   0.133 0.235
      0.80      0.86      1.000   0.067 0.125
      0.85      0.85      0.000   0.000 0.000
      0.90      0.85      0.000   0.000 0.000
      0.95      0.85      0.000   0.000 0.000
```

**Selected thresholds (Task 5):** f1_optimal 0.25, safety_team 0.15, call_center 0.3.

**Operating points on validation (Task 6):**

```
operating_point  threshold  accuracy  precision  recall    f1
     f1_optimal       0.25      0.97      0.875   0.933 0.903
    safety_team       0.15      0.86      0.517   1.000 0.682
    call_center       0.30      0.96      1.000   0.733 0.846
```

**Final report at the locked threshold (Task 7):**

```
     split  threshold  accuracy  precision  recall    f1
validation       0.25      0.97      0.875   0.933 0.903
      test       0.25      0.98      1.000   0.867 0.929
```

## Part 2: The three-way split returns

### Task 1: split_corpus_three_way

Lab 1.2 ran on a two-way split because it tuned nothing. This lab tunes all afternoon: the threshold sweep, the stakeholder picks, and the F1-optimal choice are all decisions made by looking at metric numbers, and every one of those looks consumes a dataset. If those looks consume the test set, the final number stops predicting production behavior, because the cutoff was chosen to flatter exactly those 100 tickets. So the validation set is back, as the designated place for tuning decisions to land, and the test set goes back in the drawer until Task 7, where it gets opened exactly once.

Proportions today are 60-20-20 rather than the morning's 70-15-15: with a whole sweep to read off the validation set, the extra 25 validation tickets buy meaningfully steadier curves. Same two-call pattern as Lab 1.1: peel off 40 percent, halve it.

In [ ]:
def split_corpus_three_way(df: pd.DataFrame, seed: int = RANDOM_SEED):
    """Split the corpus 60-20-20 into train, validation, and test, stratified.

    Steps:
      1. Extract X from the "text" column and y from the "label" column,
         both as numpy arrays via .to_numpy().
      2. First train_test_split call: test_size=0.40, stratify on y,
         random_state=seed. Keep the 60 percent side as train; the 40
         percent side is a temporary pool.
      3. Second call on the pool: test_size=0.50, stratify on the POOL's
         labels, random_state=seed. The halves are validation and test.

    Returns:
      (X_train, X_val, X_test, y_train, y_val, y_test),
      sized 300, 100, 100, 300, 100, 100.
    """
    raise NotImplementedError

In [ ]:
split_result = attempt("split_corpus_three_way", lambda: split_corpus_three_way(tickets_df))

if split_result is not None:
    X_train, X_val, X_test, y_train, y_val, y_test = split_result
    for name, Xs, ys in (("Train", X_train, y_train), ("Validation", X_val, y_val), ("Test", X_test, y_test)):
        print(f"{name:<10}: n={len(Xs)}, urgent={int(ys.sum())} ({100 * ys.mean():.1f}%)")
else:
    X_train = X_val = X_test = y_train = y_val = y_test = None

In [ ]:
check(
    "task 1: sizes are 300, 100, 100",
    lambda: len(X_train) == 300 and len(X_val) == 100 and len(X_test) == 100,
)
check(
    "task 1: stratified, 45, 15, and 15 urgent tickets per split",
    lambda: int(y_train.sum()) == 45 and int(y_val.sum()) == 15 and int(y_test.sum()) == 15,
)
check(
    "task 1: same seed reproduces the same split",
    lambda: np.array_equal(split_corpus_three_way(tickets_df)[4], y_val),
)
summary()

## Part 3: The classifier

### Task 2: build_and_fit_pipeline

Third time today, still from a blank cell: two named steps, one fit. By now the interesting part is not the code, it is the trap you can already predict. This corpus is imbalanced (15 percent urgent) and full of quiet positives, so before running the apply cell, write down your guess for how many of the 100 validation tickets the default `predict` will flag. The apply cell prints the answer.

In [ ]:
def build_and_fit_pipeline(X_train, y_train, seed: int = RANDOM_SEED) -> Pipeline:
    """Build and fit the TF-IDF plus logistic regression pipeline.

    Steps:
      1. Pipeline with two named steps:
         "tfidf": TfidfVectorizer(ngram_range=(1, 2))
         "clf":   LogisticRegression(max_iter=1000, random_state=seed)
      2. Fit the whole pipeline on X_train, y_train.

    Returns:
      The fitted Pipeline.
    """
    raise NotImplementedError

In [ ]:
alert_pipeline = attempt("build_and_fit_pipeline", lambda: build_and_fit_pipeline(X_train, y_train))

if alert_pipeline is not None:
    y_pred_val_default = alert_pipeline.predict(X_val)
    print(f"Validation tickets flagged urgent at the default 0.5: "
          f"{int(y_pred_val_default.sum())} of {len(y_pred_val_default)}")
    print(f"Actual urgent tickets in validation: {int(y_val.sum())}")
else:
    y_pred_val_default = None

In [ ]:
check(
    "task 2: pipeline has steps named tfidf and clf with the required settings",
    lambda: alert_pipeline.named_steps["tfidf"].ngram_range == (1, 2)
    and alert_pipeline.named_steps["clf"].max_iter == 1000
    and isinstance(alert_pipeline, Pipeline),
)
check(
    "task 2: pipeline is fitted and predicts one label per validation ticket",
    lambda: len(y_pred_val_default) == 100,
)
check(
    "task 2: flags exactly 7 validation tickets at the default threshold",
    lambda: int(y_pred_val_default.sum()) == 7,
)
summary()

## Part 4: Evaluation at any threshold

### Task 3: evaluate_at_threshold

Seven flags for fifteen real emergencies: the default cutoff just left eight urgent tickets, the quiet ones, in the routine queue. Before fixing that, build the instrument that measures any cutoff.

This function is the day's evaluation conventions plus the two lines from Lab 1.1's sweep: slice the positive-class probabilities, compare against a threshold you choose, and grade the resulting labels. It takes the pipeline rather than precomputed predictions because the threshold decision happens inside; everything downstream, the sweep, the operating points table, the final test report, is this function called in a loop.

In [ ]:
def evaluate_at_threshold(pipeline: Pipeline, X, y, threshold: float):
    """Grade the pipeline on (X, y) at a chosen decision threshold.

    Steps:
      1. proba = positive-class probabilities: predict_proba's column 1.
      2. y_pred = (proba >= threshold), cast to int.
      3. metrics = dict with keys "accuracy", "precision", "recall", "f1",
         each the matching sklearn score rounded to 3 decimals;
         precision, recall, and f1 take zero_division=0.
      4. cm = confusion_matrix(y, y_pred), truth first.

    Returns:
      (metrics, cm).
    """
    raise NotImplementedError

In [ ]:
default_eval = attempt(
    "evaluate_at_threshold",
    lambda: evaluate_at_threshold(alert_pipeline, X_val, y_val, 0.5),
)

if default_eval is not None:
    val_default_metrics, cm_val_default = default_eval
    print("Validation at the default threshold 0.5")
    print("Confusion matrix (rows true, columns predicted):")
    print(cm_val_default)
    for k, v in val_default_metrics.items():
        print(f"{k:<10} {v:.3f}")

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_val_default, display_labels=["routine", "urgent_safety"]
    )
    fig, ax = plt.subplots(figsize=(5.0, 4.2))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title("Validation, threshold 0.5")
    plt.tight_layout()
    plt.show()
else:
    val_default_metrics, cm_val_default = None, None

In [ ]:
check(
    "task 3: validation metrics at 0.5 match the target output",
    lambda: dict(val_default_metrics)
    == {"accuracy": 0.92, "precision": 1.0, "recall": 0.467, "f1": 0.636},
)
check(
    "task 3: validation confusion matrix at 0.5 is [[85, 0], [8, 7]]",
    lambda: cm_val_default.shape == (2, 2)
    and np.array_equal(cm_val_default, np.array([[85, 0], [8, 7]])),
)
check(
    "task 3: at threshold 0.25 the function reproduces the sweep row",
    lambda: evaluate_at_threshold(alert_pipeline, X_val, y_val, 0.25)[0]
    == {"accuracy": 0.97, "precision": 0.875, "recall": 0.933, "f1": 0.903},
)
summary()

## Part 5: The sweep

### Task 4: threshold_sweep

Lab 1.1 sampled three thresholds; a real tuning pass samples the range. Sweep 19 evenly spaced cutoffs from 0.05 to 0.95 on the **validation** set and collect one row per cutoff. Your Task 3 function does all the grading; this task is the loop and the table around it.

Read the finished table like an instrument panel: recall can only fall as the threshold rises (a higher bar removes alarms, it never adds catches), precision broadly rises as the surviving alarms become the confident ones, and F1 peaks somewhere in the interior where the two stop trading well. The provided cell after the checks plots all three curves; the plot is the same information as the table, arranged for arguing with stakeholders.

In [ ]:
def threshold_sweep(pipeline: Pipeline, X, y, thresholds=None) -> pd.DataFrame:
    """Evaluate the pipeline at each threshold and tabulate the metrics.

    Steps:
      1. If thresholds is None, default to
         np.round(np.linspace(0.05, 0.95, 19), 2).
      2. For each threshold, call evaluate_at_threshold and collect a row
         dict with keys "threshold", "accuracy", "precision", "recall",
         "f1" (the four metric values come straight from the returned
         metrics dict).
      3. Return pd.DataFrame(rows, columns=["threshold", "accuracy",
         "precision", "recall", "f1"]) so the column order is fixed.

    Returns:
      One row per threshold, in the order given.
    """
    raise NotImplementedError

In [ ]:
sweep_df = attempt("threshold_sweep", lambda: threshold_sweep(alert_pipeline, X_val, y_val))

if sweep_df is not None:
    print("Validation threshold sweep")
    print(sweep_df.to_string(index=False))

In [ ]:
# PROVIDED: the sweep as curves. Same information as the table, arranged
# for arguing with stakeholders.
if sweep_df is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.0))
    ax1.plot(sweep_df["threshold"], sweep_df["f1"], marker="o")
    ax1.set_xlabel("threshold")
    ax1.set_ylabel("F1")
    ax1.set_title("F1 vs threshold (validation)")
    ax1.grid(True, alpha=0.3)

    ax2.plot(sweep_df["threshold"], sweep_df["precision"], marker="o", label="precision")
    ax2.plot(sweep_df["threshold"], sweep_df["recall"], marker="s", label="recall")
    ax2.set_xlabel("threshold")
    ax2.set_ylabel("score")
    ax2.set_title("Precision and recall vs threshold (validation)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
check(
    "task 4: sweep has 19 rows and the fixed column order",
    lambda: sweep_df.shape == (19, 5)
    and list(sweep_df.columns) == ["threshold", "accuracy", "precision", "recall", "f1"],
)
check(
    "task 4: the 0.25 row matches the target output",
    lambda: sweep_df.loc[sweep_df["threshold"] == 0.25].iloc[0][
        ["accuracy", "precision", "recall", "f1"]
    ].tolist() == [0.97, 0.875, 0.933, 0.903],
)
check(
    "task 4: recall never increases as the threshold rises",
    lambda: sweep_df["recall"].is_monotonic_decreasing,
)
summary()

## Part 6: Thresholds for business goals

### Task 5: select_thresholds

Cordwell has two stakeholders reading your sweep, and they want opposite ends of it:

- **The safety team** cannot tolerate missed emergencies. Their floor: recall at or above 0.95. Among cutoffs that clear it, they will take the one that wastes the fewest callbacks, the highest precision.
- **The call center manager** staffs the callback queue. Her floor: precision at or above 0.90. Among cutoffs that clear it, she wants the most emergencies still caught, the highest recall.
- **You** also record the F1-optimal cutoff, the balanced-compromise candidate that treats both error types as equally costly.

Encode all three selections as one function of the sweep table. Two engineering notes in the contract: use `idxmax`, whose first-match convention makes ties deterministic (this sweep has unique winners, but the function should not depend on luck), and return plain floats so the picks print and compare cleanly.

In [ ]:
def select_thresholds(
    sweep_df: pd.DataFrame,
    recall_floor: float = 0.95,
    precision_floor: float = 0.90,
) -> dict:
    """Pick three operating thresholds from a sweep table.

    Steps:
      1. f1_optimal: the "threshold" value on the row where "f1" is
         largest (idxmax gives the first row of a tie).
      2. safety_team: among rows with recall >= recall_floor, the
         "threshold" value on the row with the largest precision.
      3. call_center: among rows with precision >= precision_floor, the
         "threshold" value on the row with the largest recall.

    Returns:
      {"f1_optimal": float, "safety_team": float, "call_center": float}
    """
    raise NotImplementedError

In [ ]:
picks = attempt("select_thresholds", lambda: select_thresholds(sweep_df))

if picks is not None:
    for name, thr in picks.items():
        row = sweep_df.loc[sweep_df["threshold"] == thr].iloc[0]
        print(f"{name:<12} threshold={thr:.2f}  "
              f"precision={row['precision']:.3f}  recall={row['recall']:.3f}  f1={row['f1']:.3f}")

In [ ]:
_fx_sweep = pd.DataFrame({
    "threshold": [0.1, 0.2, 0.3, 0.4],
    "accuracy": [0.5, 0.6, 0.9, 0.8],
    "precision": [0.40, 0.60, 0.95, 0.97],
    "recall": [1.00, 0.96, 0.90, 0.50],
    "f1": [0.50, 0.70, 0.92, 0.66],
})

check(
    "task 5: correct picks on a hand-checkable fixture sweep",
    lambda: select_thresholds(_fx_sweep)
    == {"f1_optimal": 0.3, "safety_team": 0.2, "call_center": 0.3},
)
check(
    "task 5: real picks match the target output",
    lambda: dict(picks) == {"f1_optimal": 0.25, "safety_team": 0.15, "call_center": 0.3},
)
check(
    "task 5: returns plain floats",
    lambda: all(type(v) is float for v in picks.values()),
)
summary()

## Part 7: The operating points, side by side

### Task 6: build_operating_points_table

Three named cutoffs are three different products built from one model. Put them in one table on the **validation** set so the trade is visible in rows: the safety pick buys perfect recall with 14 callbacks that go nowhere, the call center pick buys silence-free precision while four emergencies wait in the routine queue, and the F1 pick splits the difference. Your Task 3 does the grading; this task shapes the argument.

In [ ]:
def build_operating_points_table(pipeline: Pipeline, X, y, named_thresholds) -> pd.DataFrame:
    """Tabulate metrics for a list of named operating thresholds.

    Args:
      named_thresholds: list of (name, threshold) pairs.

    Steps:
      1. For each pair, call evaluate_at_threshold and collect a row dict
         with keys "operating_point" (the name), "threshold", and the four
         metric keys from the returned dict.
      2. Return pd.DataFrame(rows, columns=["operating_point", "threshold",
         "accuracy", "precision", "recall", "f1"]), rows in the order given.
    """
    raise NotImplementedError

In [ ]:
op_table = attempt(
    "build_operating_points_table",
    lambda: build_operating_points_table(
        alert_pipeline, X_val, y_val, list(picks.items())
    ),
)

if op_table is not None:
    print("Operating points on validation")
    print(op_table.to_string(index=False))

In [ ]:
# PROVIDED: the three operating points as confusion matrices, side by side.
if op_table is not None and picks is not None:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    for ax, (name, thr) in zip(axes, picks.items()):
        _, cm = evaluate_at_threshold(alert_pipeline, X_val, y_val, thr)
        ConfusionMatrixDisplay(
            confusion_matrix=cm, display_labels=["routine", "urgent"]
        ).plot(ax=ax, colorbar=False)
        ax.set_title(f"{name} (thr={thr:.2f})")
    plt.tight_layout()
    plt.show()

In [ ]:
check(
    "task 6: table has 3 rows and the fixed column order",
    lambda: op_table.shape == (3, 6)
    and list(op_table.columns)
    == ["operating_point", "threshold", "accuracy", "precision", "recall", "f1"],
)
check(
    "task 6: safety_team row matches the target output",
    lambda: op_table.loc[op_table["operating_point"] == "safety_team"].iloc[0].tolist()
    == ["safety_team", 0.15, 0.86, 0.517, 1.0, 0.682],
)
check(
    "task 6: rows keep the order given",
    lambda: op_table["operating_point"].tolist() == ["f1_optimal", "safety_team", "call_center"],
)
summary()

## Part 8: Lock it in, grade it once

### Task 7: final_test_report

Everything so far read the validation set, which means everything so far is, in the strict sense, tuned. The number Cordwell can put in a design review has to come from data none of your choices ever saw. Lock the F1-optimal threshold, open the test drawer, grade once, and report validation and test side by side at the same cutoff.

The gap between the two rows is the honest headline. Small movement is sampling noise on 100 tickets (each urgent ticket is 0.067 of recall); a large gap would mean the threshold was overfitted to validation quirks. Interpreting that gap, not the test score itself, is the skill this task exists to build.

In [ ]:
def final_test_report(pipeline: Pipeline, X_val, y_val, X_test, y_test, threshold: float) -> pd.DataFrame:
    """Grade validation and test at one locked threshold, side by side.

    Steps:
      1. Call evaluate_at_threshold twice: once on the validation data,
         once on the test data, both at the same locked threshold.
      2. Collect one row per split: keys "split" ("validation" or "test"),
         "threshold", and the four metric keys.
      3. Return pd.DataFrame(rows, columns=["split", "threshold",
         "accuracy", "precision", "recall", "f1"]), validation row first.
    """
    raise NotImplementedError

In [ ]:
final_report = attempt(
    "final_test_report",
    lambda: final_test_report(
        alert_pipeline, X_val, y_val, X_test, y_test, picks["f1_optimal"]
    ),
)

if final_report is not None:
    print("Final report at the locked threshold")
    print(final_report.to_string(index=False))

In [ ]:
check(
    "task 7: report has 2 rows and the fixed column order",
    lambda: final_report.shape == (2, 6)
    and list(final_report.columns)
    == ["split", "threshold", "accuracy", "precision", "recall", "f1"],
)
check(
    "task 7: validation row matches the target output",
    lambda: final_report.iloc[0].tolist() == ["validation", 0.25, 0.97, 0.875, 0.933, 0.903],
)
check(
    "task 7: test row matches the target output",
    lambda: final_report.iloc[1].tolist() == ["test", 0.25, 0.98, 1.0, 0.867, 0.929],
)
summary()

## Part 9: Wrap-up discussion and reflection

**Group discussion, tables of 3 to 4, ten minutes.** Argue from the tables on your own screens:

- The default 0.5 scored F1 0.636 on validation; a five-minute sweep found 0.903 at threshold 0.25 with no retraining at all. Where did that improvement come from, and what does it say about treating 0.5 as anything more than a placeholder?
- The safety team's cutoff catches all 15 emergencies at the cost of 14 pointless callbacks; the call center's catches 11 with zero false alarms. Neither is wrong. What information, missing from every table you built today, would settle which one Cordwell should run? (You will compute exactly that information in stretch goal 3.)
- On the final report, test precision came out higher than validation precision and test recall came out lower, at the same threshold on same-sized splits. One teammate claims the threshold is overfitted; another claims it generalized fine. Who has the better case, and what single number in the two rows do they each point at?

**Reflection, individually, 3 to 5 sentences in the cell below.** Cover: how the threshold moved precision, recall, and F1 across your sweep; which cutoff you would recommend to each stakeholder and why; and what you would want to measure before locking any of them in production.

*Write your reflection here.*

## Stretch goals (fast finishers)

Solutions live in the instructor solution notebook, released after the lab. Both hint files cover these at the same tiered depth as the main tasks.

**Stretch 1: Ship the cutoff with the model.** Your locked threshold currently lives in a notebook variable; a downstream service calling plain `predict` would silently get 0.5. Wrap the fitted pipeline in `FixedThresholdClassifier` (with `FrozenEstimator`, both from this morning's Lab 1.1 stretch) at the locked 0.25 and verify its plain `predict` reproduces your Task 7 test row exactly.

**Stretch 2: The sweep's continuous twin.** Your 19-row sweep is a discrete sampling of a curve sklearn can draw exactly: `precision_recall_curve` on the validation probabilities, summarized by `average_precision_score`. Plot the curve and mark your three operating points on it. Then compare the average precision here to the 1.000 that Lab 1.2's stretch found, and account for the difference: what is true of this corpus's probability scores that was not true there?

**Stretch 3: The number the discussion was missing.** F1 treats a false alarm and a missed emergency as equally costly, which is itself a business assumption nobody signed off on. Suppose a pointless callback costs Cordwell 8 dollars of coordinator time and a missed emergency costs an expected 400 dollars in damage and escalation. For every threshold in your sweep, compute expected cost from the confusion matrix (FP times 8 plus FN times 400), find the cost-minimizing cutoff, and compare it to the three picks from Task 5. Whose instinct did the arithmetic vindicate?

## Recap

- A trained classifier is not a decision system until someone chooses the cutoff, and 0.5 is a placeholder, not a choice. The default left 8 of 15 emergencies unflagged; moving one number to 0.25 recovered 6 of them without touching the model.
- The sweep is the instrument: recall falls monotonically as the bar rises, precision broadly climbs, and F1 peaks where the trade stops paying. Every operating decision today was a row read off that table, which is what made each decision defensible.
- Different stakeholders legitimately want different rows. Recall floors and precision floors are how business constraints become filters on a sweep, and the argument between the safety team and the call center is settled by costs, not by metrics alone.
- Tuning consumes data. Every threshold choice was made against validation; the test set was opened exactly once, at the locked cutoff, and the small validation-to-test gap is what an honest generalization story looks like on 100-ticket splits.
- The day in one line: Lab 1.1 taught what the four numbers mean, Lab 1.2 taught which number deserves to drive on imbalanced data, and this lab taught where the human decision enters, because the threshold is a business choice wearing a float's clothing.

In [ ]:
summary()
if CHECK_RESULTS and all(CHECK_RESULTS.values()):
    print("All checks passing. Lab complete.")